In [2]:
BRANCH = "kaggle"

In [3]:
!pip install dagshub dvc faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 82.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import os
import dagshub
import traceback
#from google.colab import userdata
from kaggle_secrets import UserSecretsClient
import pandas as pd
import numpy as np
import re
from dagshub import get_repo_bucket_client
import io

In [5]:
!pip install -U s3fs aiobotocore botocore --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.0/102.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 88.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.7.0 which is incompatible.


In [21]:
client = get_repo_bucket_client("dilminidasunika/movie_recommendation_system", flavor="boto")
obj = client.get_object(
    Bucket="movie_recommendation_system",
    Key="data/dataset/TMDB_all_movies.csv",
)
df = pd.read_csv(io.BytesIO(obj["Body"].read()))
df.head()

Client created. Use the name of the repo (movie_recommendation_system) as the name of the bucket

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,...,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path,keywords,certification_us
0,2,Ariel,7.100,382.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,...,Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Aki Kaurismäki,NaN,7.4,9999.0,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,"prison|underdog|helsinki, finland|factory work...",NaN
1,3,Shadows in Paradise,7.300,448.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,...,Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Mika Kaurismäki,NaN,7.4,8840.0,/nj01hspawPof0mJmlgfjuLyJuRN.jpg,"helsinki, finland|salesclerk|garbage",NR
2,5,Four Rooms,5.905,2851.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,...,"Quentin Tarantino, Robert Rodriguez, Alexandre...","Phil Parmet, Guillermo Navarro, Rodrigo García...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Lawrence Bender, Alexandre ...",Combustible Edison,6.7,117462.0,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg,hotel|new year's eve|witch|bet|sperm|hotel roo...,R
3,6,Judgment Night,6.500,385.0,Released,1993-10-15,12136938.0,110.0,21000000.0,tt0107286,...,Stephen Hopkins,Peter Levy,"Lewis Colick, Jere Cunningham","Gene Levy, Marilyn Vance, Lloyd Segan",Alan Silvestri,6.6,21307.0,/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg,"drug dealer|chicago, illinois|escape|one night...",R
4,8,Life in Loops (A Megacities RMX),7.300,34.0,Released,2006-01-01,0.0,80.0,42000.0,tt0825671,...,Timo Novotny,Wolfgang Thaler,"Michael Glawogger, Timo Novotny","Ulrich Gehmacher, Timo Novotny",NaN,8.1,285.0,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg,megacities,NaN


In [13]:
# # Step 1: Install and authenticate
# dagshub.auth.add_app_token(secret_value_0)

# # Step 2: Mount the storage bucket (Linux/Colab supports this)
# mount_path = str(dagshub.storage.mount("dilminidasunika/movie_recommendation_system"))

# print("Mounted at:", mount_path)
# # Now you can read files directly from mount_path

In [14]:
# # ../data/TMDB_all_movies.csv
# df = pd.read_csv( mount_path + "/data/dataset/TMDB_all_movies.csv")

# df

In [15]:
df.T

,0,1,2,3,4,5,6,7,8,9,...,1237102,1237103,1237104,1237105,1237106,1237107,1237108,1237109,1237110,1237111
id,2,3,5,6,8,9,11,12,13,14,...,1757475,1757476,1757477,1757481,1757482,1757483,1757484,1757485,1757486,1757487
title,Ariel,Shadows in Paradise,Four Rooms,Judgment Night,Life in Loops (A Megacities RMX),Sunday in August,Star Wars,Finding Nemo,Forrest Gump,American Beauty,...,Miss Marple : Endless Night,Tempestad en la Cordillera,Finger Guns,Lab Pe Aati Hai Dua by Syeda Areeba Fatima,El Guardián,FIGHT,Ahma's Favourite Tree,Between Silence and Truth,Mimmo the Milker,Usunure Panty
vote_average,7.1,7.3,5.905,6.5,7.3,6.8,8.207,7.82,8.463,7.998,...,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
vote_count,382.0,448.0,2851.0,385.0,34.0,28.0,22739.0,20878.0,30311.0,13270.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
status,Released,Released,Released,Released,Released,Released,Released,Released,Released,Released,...,Released,Released,Released,Released,Released,Released,Released,Released,Released,Released
release_date,1988-10-21,1986-10-17,1995-12-09,1993-10-15,2006-01-01,2004-09-02,1977-05-25,2003-05-30,1994-06-23,1999-09-15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
revenue,0.0,0.0,4257354.0,12136938.0,0.0,0.0,775398007.0,940335536.0,677387716.0,356296601.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
runtime,73.0,74.0,98.0,110.0,80.0,15.0,121.0,100.0,142.0,122.0,...,0.0,0.0,10.0,4.0,0.0,0.0,0.0,0.0,8.0,60.0
budget,0.0,0.0,4000000.0,21000000.0,42000.0,0.0,11000000.0,94000000.0,55000000.0,15000000.0,...,0.0,2500.0,0.0,0.0,200.0,0.0,0.0,0.0,0.0,0.0
imdb_id,tt0094675,tt0092149,tt0113101,tt0107286,tt0825671,tt0425473,tt0076759,tt0266543,tt0109830,tt0169547,...,tt2166494,tt33070889,tt44792194,tt44878814,NaN,NaN,NaN,tt34725314,NaN,tt4450898


# Select features

Plot
- overview

Tags
- title
- original_title
- original_language
- genres
- production_companies
- production_countries
- spoken_languages
- cast
- director
- writers
- producers
- music_composer
- keywords



# Preprocessing

In [22]:
status = (
    df['status']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique status:", status.nunique())
print(status.value_counts())

Number of unique status: 6
status
Released           1213515
Post Production       7479
In Production         6954
Planned               6746
Rumored               1224
Canceled              1194
Name: count, dtype: int64


In [23]:
df = df[df['status'] == 'Released'].reset_index(drop=True)
df.shape

(1213515, 30)

In [24]:
selected_df = df[["title", "original_title", "release_date", "original_language", "genres", "production_companies", "production_countries", "spoken_languages", "cast", "director", "writers", "producers", "music_composer", "keywords", "overview", "poster_path"]]
selected_df

,title,original_title,release_date,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords,overview,poster_path
0,Ariel,Ariel,1988-10-21,fi,"Drama, Romance, Crime",Villealfa Filmproductions,Finland,suomi,"Turo Pajala, Susanna Haavisto, Matti Pellonpää...",Aki Kaurismäki,Aki Kaurismäki,Aki Kaurismäki,NaN,"prison|underdog|helsinki, finland|factory work...",A Finnish man goes to the city to find a job a...,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,Shadows in Paradise,Varjoja paratiisissa,1986-10-17,fi,"Drama, Romance",Villealfa Filmproductions,Finland,"svenska, suomi, English","Matti Pellonpää, Kati Outinen, Sakari Kuosmane...",Aki Kaurismäki,Aki Kaurismäki,Mika Kaurismäki,NaN,"helsinki, finland|salesclerk|garbage","Nikander, a rubbish collector and would-be ent...",/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,Four Rooms,Four Rooms,1995-12-09,en,Comedy,"Miramax, A Band Apart",United States of America,English,"Sammi Davis, Salma Hayek Pinault, Lawrence Ben...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Lawrence Bender, Alexandre ...",Combustible Edison,hotel|new year's eve|witch|bet|sperm|hotel roo...,It's Ted the Bellhop's first night on the job....,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,Judgment Night,Judgment Night,1993-10-15,en,"Action, Crime, Drama, Thriller","Largo Entertainment, JVC, Universal Pictures",United States of America,English,"Emilio Estevez, Cuba Gooding Jr., Denis Leary,...",Stephen Hopkins,"Lewis Colick, Jere Cunningham","Gene Levy, Marilyn Vance, Lloyd Segan",Alan Silvestri,"drug dealer|chicago, illinois|escape|one night...","Four young friends, while taking a shortcut en...",/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,Life in Loops (A Megacities RMX),Life in Loops (A Megacities RMX),2006-01-01,en,Documentary,inLoops,Austria,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,"Michael Glawogger, Timo Novotny","Ulrich Gehmacher, Timo Novotny",NaN,megacities,Timo Novotny labels his new project an experim...,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1213510,FIGHT,FIGHT,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1213511,Ahma's Favourite Tree,Ahma's Favourite Tree,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1213512,Between Silence and Truth,Between Silence and Truth,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1213513,Mimmo the Milker,Mimmo the Milker,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Alien cows, hungry for cheddar, invade Planet ...",NaN


In [25]:
selected_df.shape

(1213515, 16)

In [26]:
print("overview empty count:" , int(selected_df["overview"].isna().sum()))
print("title empty count:" , int(selected_df["title"].isna().sum()))
print("original_title empty count:" , int(selected_df["original_title"].isna().sum()))
print("release_date empty count:" , int(selected_df["release_date"].isna().sum()))
print("original_language empty count:" , int(selected_df["original_language"].isna().sum()))
print("genres empty count:" , int(selected_df["genres"].isna().sum()))
print("production_companies empty count:" , int(selected_df["production_companies"].isna().sum()))
print("production_countries empty count:" , int(selected_df["production_countries"].isna().sum()))
print("spoken_languages empty count:" , int(selected_df["spoken_languages"].isna().sum()))
print("cast empty count:" , int(selected_df["cast"].isna().sum()))
print("director empty count:" , int(selected_df["director"].isna().sum()))
print("writers empty count:" , int(selected_df["writers"].isna().sum()))
print("producers empty count:" , int(selected_df["producers"].isna().sum()))
print("music_composer empty count:" , int(selected_df["music_composer"].isna().sum()))
print("keywords empty count:" , int(selected_df["keywords"].isna().sum()))
print("poster_path empty count:" , int(selected_df["poster_path"].isna().sum()))

overview empty count: 184898
title empty count: 9
original_title empty count: 9
release_date empty count: 111820
original_language empty count: 0
genres empty count: 327603
production_companies empty count: 629855
production_countries empty count: 452168
spoken_languages empty count: 445325
cast empty count: 382410
director empty count: 194499
writers empty count: 602349
producers empty count: 800455
music_composer empty count: 1072333
keywords empty count: 869686
poster_path empty count: 287349


In [27]:
selected_df[(selected_df['title'].isna() & ~selected_df['original_title'].isna())]

,title,original_title,release_date,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords,overview,poster_path


## Remove empty Data rows

In [28]:
cleaned_df = selected_df.dropna(subset=['title','original_title'], how='all')
cleaned_df.shape

(1213506, 16)

In [29]:
cleaned_df = cleaned_df.dropna(subset=['genres','cast','director','writers','producers','music_composer','keywords','overview'], how='all')
cleaned_df.shape

(1192422, 16)

In [30]:
cleaned_df

,title,original_title,release_date,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords,overview,poster_path
0,Ariel,Ariel,1988-10-21,fi,"Drama, Romance, Crime",Villealfa Filmproductions,Finland,suomi,"Turo Pajala, Susanna Haavisto, Matti Pellonpää...",Aki Kaurismäki,Aki Kaurismäki,Aki Kaurismäki,NaN,"prison|underdog|helsinki, finland|factory work...",A Finnish man goes to the city to find a job a...,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,Shadows in Paradise,Varjoja paratiisissa,1986-10-17,fi,"Drama, Romance",Villealfa Filmproductions,Finland,"svenska, suomi, English","Matti Pellonpää, Kati Outinen, Sakari Kuosmane...",Aki Kaurismäki,Aki Kaurismäki,Mika Kaurismäki,NaN,"helsinki, finland|salesclerk|garbage","Nikander, a rubbish collector and would-be ent...",/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,Four Rooms,Four Rooms,1995-12-09,en,Comedy,"Miramax, A Band Apart",United States of America,English,"Sammi Davis, Salma Hayek Pinault, Lawrence Ben...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Lawrence Bender, Alexandre ...",Combustible Edison,hotel|new year's eve|witch|bet|sperm|hotel roo...,It's Ted the Bellhop's first night on the job....,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,Judgment Night,Judgment Night,1993-10-15,en,"Action, Crime, Drama, Thriller","Largo Entertainment, JVC, Universal Pictures",United States of America,English,"Emilio Estevez, Cuba Gooding Jr., Denis Leary,...",Stephen Hopkins,"Lewis Colick, Jere Cunningham","Gene Levy, Marilyn Vance, Lloyd Segan",Alan Silvestri,"drug dealer|chicago, illinois|escape|one night...","Four young friends, while taking a shortcut en...",/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,Life in Loops (A Megacities RMX),Life in Loops (A Megacities RMX),2006-01-01,en,Documentary,inLoops,Austria,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,"Michael Glawogger, Timo Novotny","Ulrich Gehmacher, Timo Novotny",NaN,megacities,Timo Novotny labels his new project an experim...,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1213504,a(ng)kin,a(ng)kin,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A young man traverses the landscapes of his ne...,NaN
1213507,Finger Guns,Finger Guns,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Noelle didn't mind being a secondary piece in ...,NaN
1213508,Lab Pe Aati Hai Dua by Syeda Areeba Fatima,Lab Pe Aati Hai Dua by Syeda Areeba Fatima,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Official music video for ""Lab Pe Aati Hai Dua""...",NaN
1213513,Mimmo the Milker,Mimmo the Milker,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Alien cows, hungry for cheddar, invade Planet ...",NaN


In [31]:
print("overview empty count:" , int(cleaned_df["overview"].isna().sum()))
print("title empty count:" , int(cleaned_df["title"].isna().sum()))
print("original_title empty count:" , int(cleaned_df["original_title"].isna().sum()))
print("original_language empty count:" , int(cleaned_df["original_language"].isna().sum()))
print("genres empty count:" , int(cleaned_df["genres"].isna().sum()))
print("production_companies empty count:" , int(cleaned_df["production_companies"].isna().sum()))
print("production_countries empty count:" , int(cleaned_df["production_countries"].isna().sum()))
print("spoken_languages empty count:" , int(cleaned_df["spoken_languages"].isna().sum()))
print("cast empty count:" , int(cleaned_df["cast"].isna().sum()))
print("director empty count:" , int(cleaned_df["director"].isna().sum()))
print("writers empty count:" , int(cleaned_df["writers"].isna().sum()))
print("producers empty count:" , int(cleaned_df["producers"].isna().sum()))
print("music_composer empty count:" , int(cleaned_df["music_composer"].isna().sum()))
print("keywords empty count:" , int(cleaned_df["keywords"].isna().sum()))
print("poster_path empty count:" , int(cleaned_df["poster_path"].isna().sum()))

overview empty count: 163810
title empty count: 0
original_title empty count: 0
original_language empty count: 0
genres empty count: 306517
production_companies empty count: 609326
production_countries empty count: 432476
spoken_languages empty count: 426988
cast empty count: 361321
director empty count: 173412
writers empty count: 581259
producers empty count: 779365
music_composer empty count: 1051241
keywords empty count: 848597
poster_path empty count: 276564


## Check Unique Tags

In [105]:
genres = (
    cleaned_df['genres']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique genres:", genres.nunique())
print(genres.value_counts())

Number of unique genres: 19
genres
Drama              295992
Documentary        233263
Comedy             176170
Animation           73277
Horror              71609
Music               65221
Romance             63914
Thriller            59894
Action              52615
Crime               41979
Family              35002
TV Movie            33416
Fantasy             30005
Adventure           28103
Mystery             26797
Science Fiction     26763
History             22320
War                 13112
Western             10135
Name: count, dtype: int64


In [34]:
company = (
    cleaned_df['production_companies']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique companies:", company.nunique())
print(company.value_counts())

Number of unique companies: 213277
production_companies
BBC                      4147
ARTE                     3719
ONF | NFB                3296
Warner Bros. Pictures    3064
Columbia Pictures        2978
                         ... 
RSTV                        1
Sineva Film                 1
Festus' Film Factory        1
ADN Digital                 1
Emu Films                   1
Name: count, Length: 213277, dtype: int64


In [35]:
countries = (
    cleaned_df['production_countries']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique countries:", countries.nunique())
print(countries.value_counts())

Number of unique countries: 250
production_countries
United States of America    187543
France                       56602
United Kingdom               53945
Japan                        50571
Germany                      47673
                             ...  
Anguilla                         2
Cocos  Islands                   2
East Timor                       2
Burma                            2
Zaire                            1
Name: count, Length: 250, dtype: int64


In [36]:
original_language = (
    cleaned_df['original_language']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique original_language:", original_language.nunique())
print(original_language.value_counts())

Number of unique original_language: 178
original_language
en    584704
fr     75444
es     72907
ja     56650
de     53108
       ...  
io         1
nr         1
ii         1
ng         1
pi         1
Name: count, Length: 178, dtype: int64


In [37]:
spoken_languages = (
    cleaned_df['spoken_languages']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique spoken_languages:", spoken_languages.nunique())
print(spoken_languages.value_counts())

Number of unique spoken_languages: 92
spoken_languages
English          290062
Español           62899
Français          61793
日本語               54073
No Language       48171
                  ...  
Qafar af              5
Kirundi               5
Aragonés              5
Bislama               2
Upastawakaēna         2
Name: count, Length: 92, dtype: int64


In [32]:
def split_cast(s):
    parts = re.split(r',(?!\s*(?:Jr|jr|Sr|sr|II|III|IV|V)\.?\s*(?:,|$))\s*', s)
    return [p.strip() for p in parts if p.strip()]

In [39]:
cast = (
    cleaned_df['cast']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique cast:", cast.nunique())
print(cast.value_counts())

Number of unique cast: 2188772
cast
Jr.                    1210
Mel Blanc              1124
Bess Flowers            751
Jagathy Sreekumar       686
Brahmanandam            678
                       ... 
Evis Sánchez              1
Kelsi Cook                1
Erin Rose Doyle           1
Alex Joseph Grayson       1
Ayhan Polat               1
Name: count, Length: 2188772, dtype: int64


In [40]:
cast2 = cleaned_df['cast'].dropna().apply(split_cast).explode().str.strip()

print("Number of unique cast:", cast2.nunique())
print(cast2.value_counts())
suspects = cast2.value_counts()[cast2.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', ''])]
print(suspects)

Number of unique cast: 2188903
cast
Mel Blanc            1124
Bess Flowers          751
Jagathy Sreekumar     686
Brahmanandam          678
Eric Roberts          674
                     ... 
Ayhan Polat             1
Santiago Damilano       1
Cuba Betancourt         1
Brandon Gomez           1
Anastasio Velasco       1
Name: count, Length: 2188903, dtype: int64
cast
Jr.    1
Name: count, dtype: int64


In [41]:
director = (
    cleaned_df['director']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique director:", director.nunique())
print(director.value_counts())
suspects_dir = director.value_counts()[director.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_dir)

Number of unique director: 434820
director
Dave Fleischer        639
Gérard Courant        502
D.W. Griffith         467
Kevin Dunn            464
Paul Terry            413
                     ... 
Pauline Zufferey        1
Hugo Cousino            1
Hendrik S. Schmitt      1
Jazz Pitcairn           1
Martín Florio           1
Name: count, Length: 434820, dtype: int64
director
Jr.    171
        19
Sr.      4
III      2
Jr       2
V        1
Name: count, dtype: int64


In [42]:
director2 = (
    cleaned_df['director']
    .dropna()
    .apply(split_cast)
    .explode()
    .str.strip()
)

print("Number of unique director:", director2.nunique())
print(director2.value_counts())
suspects_dir2 = director2.value_counts()[director2.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_dir2)

Number of unique director: 434820
director
Dave Fleischer          639
Gérard Courant          502
D.W. Griffith           467
Kevin Dunn              464
Paul Terry              413
                       ... 
Noémie Ferron             1
Joe Duplantier            1
Christo Clement           1
Wilma Lundberg Linde      1
Ana Butliashvili          1
Name: count, Length: 434820, dtype: int64
director
V    1
Name: count, dtype: int64


In [43]:
writers = (
    cleaned_df['writers']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique writers:", writers.nunique())
print(writers.value_counts())
suspects_wri = writers.value_counts()[writers.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V','X', ''])]
print(suspects_wri)

Number of unique writers: 409225
writers
William Shakespeare     766
Stephen King            520
Anton Chekhov           351
Vince McMahon           305
Edgar Allan Poe         260
                       ... 
Jazz Pitcairn             1
CoralineHecc              1
Kyle DeNigris             1
Martín Paredes            1
Tornike Tetunashvili      1
Name: count, Length: 409225, dtype: int64
writers
Jr.    203
         9
Sr.      3
V        3
Jr       3
II       1
III      1
Name: count, dtype: int64


In [44]:
writers2 = (
    cleaned_df['writers']
    .dropna()
    .apply(split_cast)
    .explode()
    .str.strip()
)

print("Number of unique writers:", writers2.nunique())
print(writers2.value_counts())
suspects_wri2 = writers2.value_counts()[writers2.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_wri2)

Number of unique writers: 409226
writers
William Shakespeare    766
Stephen King           520
Anton Chekhov          351
Vince McMahon          305
Edgar Allan Poe        260
                      ... 
Tim Arem                 1
Paul Bossé               1
Martín Sepúlveda         1
Blake Tucker             1
Gosha Kvinikadze         1
Name: count, Length: 409226, dtype: int64
writers
V    3
Name: count, dtype: int64


In [45]:
producers = (
    cleaned_df['producers']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique producers:", producers.nunique())
print(producers.value_counts())
suspects_prod = producers.value_counts()[producers.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_prod)

Number of unique producers: 352013
producers
Louis Lumière            964
Auguste Lumière          886
Walt Disney              628
Tom Berry                611
Sebastian Battro         533
                        ... 
Riad Ouazzani Touhami      1
Joane Hétu                 1
Eric Neumann               1
Steven J. Schupak          1
Fajri Kieflie              1
Name: count, Length: 352013, dtype: int64
producers
Jr.    209
         9
Sr.      6
III      4
V        2
Jr       1
Name: count, dtype: int64


In [46]:
producers2 = (
    cleaned_df['producers']
    .dropna()
    .apply(split_cast)
    .explode()
    .str.strip()
)

print("Number of unique producers:", producers2.nunique())
print(producers2.value_counts())
suspects_prod2 = producers2.value_counts()[producers2.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_prod2)

Number of unique producers: 352022
producers
Louis Lumière       964
Auguste Lumière     886
Walt Disney         628
Tom Berry           611
Sebastian Battro    533
                   ... 
Mads Toft Hansen      1
Jake Isaacs           1
Kara Oropallo         1
Angela Lepito         1
Marisa Laurance       1
Name: count, Length: 352022, dtype: int64
Series([], Name: count, dtype: int64)


In [47]:
music_composer = (
    cleaned_df['music_composer']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

print("Number of unique music_composer:", music_composer.nunique())
print(music_composer.value_counts())
suspects_music = music_composer.value_counts()[music_composer.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_music)

Number of unique music_composer: 47253
music_composer
Ilaiyaraaja        706
Ennio Morricone    327
Masaru Satō        263
Zdeněk Liška       255
Chūji Kinoshita    249
                  ... 
Mikko Alatalo        1
Martti Luoma         1
João Donato          1
Michael Boland       1
Dorothy Cox          1
Name: count, Length: 47253, dtype: int64
music_composer
Jr.    4
Name: count, dtype: int64


In [48]:
music_composer2 = (
    cleaned_df['music_composer']
    .dropna()
    .apply(split_cast)
    .explode()
    .str.strip()
)

print("Number of unique music_composer:", music_composer2.nunique())
print(music_composer2.value_counts())
suspects_music2 = music_composer2.value_counts()[music_composer2.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', ''])]
print(suspects_music2)

Number of unique music_composer: 47252
music_composer
Ilaiyaraaja        706
Ennio Morricone    327
Masaru Satō        263
Zdeněk Liška       255
Chūji Kinoshita    249
                  ... 
Finlay White         1
Dan Persad           1
Murat Engin          1
Sigrid Kemp          1
Tural Bakuvi         1
Name: count, Length: 47252, dtype: int64
Series([], Name: count, dtype: int64)


In [49]:
keywords = (
    cleaned_df['keywords']
    .dropna()
    .str.split('|')
    .explode()
    .str.strip()
)

print("Number of unique keywords:", keywords.nunique())
print(keywords.value_counts())
suspects_key = keywords.value_counts()[keywords.value_counts().index.isin(['Jr.', 'Jr', 'Sr.', 'Sr', 'II', 'III', 'IV', 'V', ''])]
print(suspects_key)

Number of unique keywords: 76543
keywords
short film                37752
woman director            16242
based on novel or book     7807
lgbt                       7110
concert                    6181
                          ...  
comportment                   1
homme                         1
commissaires                  1
artistes                      1
histoire de l'art             1
Name: count, Length: 76543, dtype: int64
Series([], Name: count, dtype: int64)


In [33]:
def has_lone_jr(s):
    parts = split_cast(s)
    return any(p.strip() in ('Jr.', 'Jr', 'jr.', 'jr') for p in parts)

offenders = cleaned_df['cast'].dropna()
offenders = offenders[offenders.apply(has_lone_jr)]
print(offenders.head(20).tolist())

['Jr.']


In [34]:
mask = cleaned_df['cast'].str.contains(r'\bjr\b', case=False, na=False, regex=True)
print(mask.sum())   # how many rows actually matched — do this FIRST, always

17009


In [35]:
with pd.option_context('display.max_colwidth', None):
    display(cleaned_df.loc[mask, 'cast'].sample(20))

316384                                                                                    John Travolta, Katheryn Winnick, Jennifer Esposito, Matthew Modine, James Remar, Kellan Lutz, Moran Atias, Jordi Mollà, Michael Weston, Amaury Nolasco, Mariana Paola Vicente, Luis Da Silva, Jr., Ed Amatrudo, Keith Hudson, Rob Gough, Stewart J. Zully, Amber Rivera, Justin Nesbitt, Omar Cruz Soto, Charlie Gillespie, Anita Leeman, Prashant Shah, Denny Mendez, Carlos Alberto López, Luis Rosado, Bruno Irizarry, Jeffrey Holsman, Samuel Jove, Isabella Rivera-Gandulla, Craig Chamberlin, Eduardo Burgos, Jean Pierre Prats, Tom Sizemore
92568                                                                                                                                                                                                                                                                                                                                                                                     

In [53]:
# cleaned_df.to_csv(mount_path + "/data/updatedmovies.csv", index=False)

## Test

In [ ]:
test_df = pd.DataFrame({
    'cast': [
        'Robert Downey, Jr., Tom Hanks, Samuel L. Jackson III',
        'Meryl Streep, Denzel Washington',
        np.nan,
        '',
        'Martin Luther King, Sr.',
        'Will Smith'
    ],
    'genres': [
        'Action, Adventure',
        'Drama',
        np.nan,
        '',
        'Comedy, Romance',
        np.nan
    ]
})

test_df

,cast,genres
0,"Robert Downey, Jr., Tom Hanks, Samuel L. Jacks...","Action, Adventure"
1,"Meryl Streep, Denzel Washington",Drama
2,NaN,NaN
3,,
4,"Martin Luther King, Sr.","Comedy, Romance"
5,Will Smith,NaN


In [ ]:
# --- BEFORE: check dtype ---
print(test_df['cast'].dtype)           # object
print(type(test_df['cast'].iloc[0]))   # <class 'str'>
print(type(test_df['cast'].iloc[2]))   # <class 'float'> (NaN)

# --- CONVERT using split_cast ---
cast = test_df['cast'].fillna('').apply(split_cast)

# --- AFTER: check dtype ---
print(cast.dtype)             # still object
print(type(cast.iloc[0]))     # <class 'list'>
print(type(cast.iloc[2]))     # <class 'list'> -- was NaN, now []

# --- INSPECT ROW BY ROW ---
for i in range(len(test_df)):
    print(i, '|', repr(test_df['cast'].iloc[i]), '->', cast.iloc[i])

# --- SANITY CHECK ---
assert cast.apply(lambda x: isinstance(x, list)).all()
print("all rows are lists ✓")

object
<class 'str'>
<class 'float'>
object
<class 'list'>
<class 'list'>
0 | 'Robert Downey, Jr., Tom Hanks, Samuel L. Jackson III' -> ['Robert Downey, Jr.', 'Tom Hanks', 'Samuel L. Jackson III']
1 | 'Meryl Streep, Denzel Washington' -> ['Meryl Streep', 'Denzel Washington']
2 | nan -> []
3 | '' -> []
4 | 'Martin Luther King, Sr.' -> ['Martin Luther King, Sr.']
5 | 'Will Smith' -> ['Will Smith']
all rows are lists ✓


In [ ]:
cast

,cast
0,"[Robert Downey, Jr., Tom Hanks, Samuel L. Jack..."
1,"[Meryl Streep, Denzel Washington]"
2,[]
3,[]
4,"[Martin Luther King, Sr.]"
5,[Will Smith]


## Check Missing values

In [15]:
# cleaned_df = pd.read_csv(mount_path + "/data/updatedmovies.csv")

In [7]:
client = get_repo_bucket_client("dilminidasunika/movie_recommendation_system", flavor="boto")
obj = client.get_object(
    Bucket="movie_recommendation_system",
    Key="data/updatedmovies.csv",
)
cleaned_df = pd.read_csv(io.BytesIO(obj["Body"].read()))
cleaned_df.head()

Client created. Use the name of the repo (movie_recommendation_system) as the name of the bucket

,title,original_title,release_date,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords,overview,poster_path
0,Ariel,Ariel,1988-10-21,fi,"Drama, Romance, Crime",Villealfa Filmproductions,Finland,suomi,"Turo Pajala, Susanna Haavisto, Matti Pellonpää...",Aki Kaurismäki,Aki Kaurismäki,Aki Kaurismäki,NaN,"prison|underdog|helsinki, finland|factory work...",A Finnish man goes to the city to find a job a...,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,Shadows in Paradise,Varjoja paratiisissa,1986-10-17,fi,"Drama, Romance",Villealfa Filmproductions,Finland,"svenska, suomi, English","Matti Pellonpää, Kati Outinen, Sakari Kuosmane...",Aki Kaurismäki,Aki Kaurismäki,Mika Kaurismäki,NaN,"helsinki, finland|salesclerk|garbage","Nikander, a rubbish collector and would-be ent...",/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,Four Rooms,Four Rooms,1995-12-09,en,Comedy,"Miramax, A Band Apart",United States of America,English,"Sammi Davis, Salma Hayek Pinault, Lawrence Ben...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Lawrence Bender, Alexandre ...",Combustible Edison,hotel|new year's eve|witch|bet|sperm|hotel roo...,It's Ted the Bellhop's first night on the job....,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,Judgment Night,Judgment Night,1993-10-15,en,"Action, Crime, Drama, Thriller","Largo Entertainment, JVC, Universal Pictures",United States of America,English,"Emilio Estevez, Cuba Gooding Jr., Denis Leary,...",Stephen Hopkins,"Lewis Colick, Jere Cunningham","Gene Levy, Marilyn Vance, Lloyd Segan",Alan Silvestri,"drug dealer|chicago, illinois|escape|one night...","Four young friends, while taking a shortcut en...",/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,Life in Loops (A Megacities RMX),Life in Loops (A Megacities RMX),2006-01-01,en,Documentary,inLoops,Austria,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,"Michael Glawogger, Timo Novotny","Ulrich Gehmacher, Timo Novotny",NaN,megacities,Timo Novotny labels his new project an experim...,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg


In [8]:
cleaned_df

,title,original_title,release_date,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords,overview,poster_path
0,Ariel,Ariel,1988-10-21,fi,"Drama, Romance, Crime",Villealfa Filmproductions,Finland,suomi,"Turo Pajala, Susanna Haavisto, Matti Pellonpää...",Aki Kaurismäki,Aki Kaurismäki,Aki Kaurismäki,NaN,"prison|underdog|helsinki, finland|factory work...",A Finnish man goes to the city to find a job a...,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,Shadows in Paradise,Varjoja paratiisissa,1986-10-17,fi,"Drama, Romance",Villealfa Filmproductions,Finland,"svenska, suomi, English","Matti Pellonpää, Kati Outinen, Sakari Kuosmane...",Aki Kaurismäki,Aki Kaurismäki,Mika Kaurismäki,NaN,"helsinki, finland|salesclerk|garbage","Nikander, a rubbish collector and would-be ent...",/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,Four Rooms,Four Rooms,1995-12-09,en,Comedy,"Miramax, A Band Apart",United States of America,English,"Sammi Davis, Salma Hayek Pinault, Lawrence Ben...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Lawrence Bender, Alexandre ...",Combustible Edison,hotel|new year's eve|witch|bet|sperm|hotel roo...,It's Ted the Bellhop's first night on the job....,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,Judgment Night,Judgment Night,1993-10-15,en,"Action, Crime, Drama, Thriller","Largo Entertainment, JVC, Universal Pictures",United States of America,English,"Emilio Estevez, Cuba Gooding Jr., Denis Leary,...",Stephen Hopkins,"Lewis Colick, Jere Cunningham","Gene Levy, Marilyn Vance, Lloyd Segan",Alan Silvestri,"drug dealer|chicago, illinois|escape|one night...","Four young friends, while taking a shortcut en...",/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,Life in Loops (A Megacities RMX),Life in Loops (A Megacities RMX),2006-01-01,en,Documentary,inLoops,Austria,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,"Michael Glawogger, Timo Novotny","Ulrich Gehmacher, Timo Novotny",NaN,megacities,Timo Novotny labels his new project an experim...,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1192417,a(ng)kin,a(ng)kin,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A young man traverses the landscapes of his ne...,NaN
1192418,Finger Guns,Finger Guns,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Noelle didn't mind being a secondary piece in ...,NaN
1192419,Lab Pe Aati Hai Dua by Syeda Areeba Fatima,Lab Pe Aati Hai Dua by Syeda Areeba Fatima,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Official music video for ""Lab Pe Aati Hai Dua""...",NaN
1192420,Mimmo the Milker,Mimmo the Milker,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Alien cows, hungry for cheddar, invade Planet ...",NaN


In [9]:
cleaned_df["release_date"] = pd.to_datetime(cleaned_df["release_date"])
cleaned_df.dtypes

title                           object
original_title                  object
release_date            datetime64[ns]
original_language               object
genres                          object
production_companies            object
production_countries            object
spoken_languages                object
cast                            object
director                        object
writers                         object
producers                       object
music_composer                  object
keywords                        object
overview                        object
poster_path                     object
dtype: object

In [10]:
missing_strings = [
    "NaN", "None", "none",
    "null", "NULL",
    "N/A", "NA",
    "unknown", "Unknown", "NaT"
]

for value in missing_strings:
    count = (cleaned_df == value).sum().sum()
    if count > 0:
        print(f'"{value}": {count}')

"unknown": 2
"Unknown": 49


In [11]:
for value in ["unknown", "Unknown"]:
    counts = (cleaned_df == value).sum()
    print(f'\n"{value}":')
    print(counts[counts > 0])


"unknown":
title             1
original_title    1
dtype: int64

"Unknown":
title                   21
original_title          11
production_companies     1
cast                     4
director                 7
writers                  2
producers                3
dtype: int64


In [12]:
cleaned_df[
    (cleaned_df["title"] == "Unknown")
]

,title,original_title,release_date,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords,overview,poster_path
4970,Unknown,Unknown,2006-11-03,en,"Mystery, Thriller","Rick Lashbrook Films, Eleven Eleven Films",United States of America,English,"Jim Caviezel, Greg Kinnear, Bridget Moynahan, ...",Simón Brand,Matthew Waynee,"Tamara Delano, Randy Delano, Cedric Jeanson, R...",Angelo Milli,factory|amnesia|police|ransom|kidnapping|past|...,Five men wake up in a locked-down warehouse wi...,/h6hz8LjwyYM0jrz6pxAYicwwIrj.jpg
32004,Unknown,Unknown,2011-02-16,en,"Action, Mystery, Thriller","Panda Productions, Dark Castle Entertainment, ...","United States of America, United Kingdom, Fran...","English, Deutsch, Türkçe, العربية","Liam Neeson, Diane Kruger, January Jones, Aida...",Jaume Collet-Serra,"Didier van Cauwelaert, Oliver Butcher, Stephen...","Andrew Rona, Joel Silver, Susan Downey, Peter ...","John Ottman, Alexander Rudd",hotel|taxi|coma|taxi driver|prince|briefcase|h...,"A man awakens from a coma, only to discover th...",/aXBQD515okXQZmYA89ntXMvSJSd.jpg
197211,Unknown,Unknown,1972-01-01,en,Science Fiction,Sheffield University Union of Students,United Kingdom,English,"Andrew Robinson Smith, Andrew Webster, Dick Pa...",NaN,NaN,NaN,NaN,NaN,The Film Unit at Sheffield University Union of...,NaN
443607,Unknown,Unknown,1972-07-11,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The Film Unit at Sheffield University Union of...,NaN
570260,Unknown,ناشناس,2007-04-01,fa,"Drama, Mystery, TV Movie",NaN,NaN,فارسی,"Soheila Razavi, Laleh Eskandari, Abdolreza Akb...",Ahmad Zarifraftar,Ahmad Zarifraftar,NaN,NaN,NaN,NaN,/6ScWRqs7mSaI9QRXhasyWMvfxr4.jpg
626084,Unknown,Unknown,2021-09-18,hu,NaN,NaN,NaN,NaN,NaN,Mariann Zászlós,NaN,NaN,NaN,NaN,A testimony of two transgender women about chi...,/7D6kKRPVkxw3FFffhthG1zOzSoa.jpg
713137,Unknown,Неизвестный,2015-10-05,ru,Documentary,NaN,NaN,NaN,NaN,Yuliya Sergina,NaN,NaN,NaN,NaN,A Russian boy who was raised by Wahhabis talks...,NaN
747396,Unknown,アンノウンッ,2017-04-07,ja,Horror,NaN,Japan,日本語,NaN,Takashi Taniguchi,NaN,NaN,NaN,found footage|psychic phenomena,A collection of horror images that illuminates...,/9rXdwWVReBa37eB22zBt5uGX4aC.jpg
756820,Unknown,미지수,2024-05-08,ko,Drama,DK Film Production,South Korea,한국어/조선말,"Ban Si-on, Park Jong-hwan, Ahn Sung-min, Kwon ...",Lee Don-ku,Lee Don-ku,NaN,김철환,NaN,Wooju brings a corpse to his lover Jisu's plac...,/yiX5ikg429FFSfjemd91r0Ixhwi.jpg
769395,Unknown,غریب,2023-05-07,fa,"Family, Drama",NaN,Iran,فارسی,"Rahim Norouzi, Pardis Pourabedini, Babak Hamid...",Mohammad Hossein Latifi,Hamed Ongha,NaN,NaN,NaN,With the beginning of the Kurdish rebellion in...,NaN


In [13]:
cleaned_df[
    (cleaned_df["cast"] == "Unknown")
]

,title,original_title,release_date,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords,overview,poster_path
99750,The Haunted Ship,The Haunted Ship,1930-04-26,en,"Animation, Comedy, Horror",Van Beuren Studios,United States of America,English,Unknown,"John Foster, Harry Bailey, Mannie Davis",NaN,"Paul Terry, Amadee J. Van Beuren",NaN,NaN,Waffles the Cat and Don the Dog explore a sunk...,NaN
865069,Venezia 2024,Venezia 2024,NaT,it,Drama,NaN,NaN,Italiano,Unknown,Emma spaventati,NaN,NaN,NaN,venezia,NaN,/hIiFpVn6z6hNb1MAdMQSOTVcQ7M.jpg
956467,We Exist Triply,We Exist Triply,NaT,en,Documentary,NaN,NaN,English,Unknown,AJ Layla,NaN,NaN,NaN,"london, england|family history|parents|nostalg...",A sock puppet explores a family history told f...,/4kpxUDuTHGcQfWPLCkVl1wHEtTp.jpg
1102060,"Hein Petersen, vom Schiffsjungen zum Matrosen","Hein Petersen, vom Schiffsjungen zum Matrosen",1917-01-01,xx,"War, Documentary",Bild- und Filmamt (BuFA),Germany,NaN,Unknown,Unknown,NaN,NaN,NaN,submarine|military service|sailor|recruitment|...,NaN,NaN


In [14]:
pd.set_option('display.max_colwidth', None)

broad_mask = cleaned_df['overview'].str.contains(
    r'DUPLICATE\s*(?i:entry|record|of)',
    case=True, na=False, regex=True
)
print(f"Rows matching pattern: {broad_mask.sum()}")

for idx, row in cleaned_df[broad_mask][['title', 'overview']].iterrows():
    print(f"[{idx}] {row['title']}\n{row['overview']}\n")

Rows matching pattern: 5
[10430] A New Journey: Live at Slane Castle
DUPLICATE ENTRY! See Celtic Woman: A New Journey

[234643] Baijin long
The first Cantonese talkie. It was produced in Shanghai (Tianyi Studio) and was based on a Cantonese opera play. DUPLICATE RECORD: See BAIJIN LONG

[531552] Portsmouth
[DUPLICATE of 'Away Aloft'] Portsmouth training

[606443] Audience Leaving Circus Carré After the Biograph Show in Amsterdam
[DUPLICATE of 'After... Circus Carré'.] While he was trying, Dickson made the same sort of movie that others had before him. This is an imitation of a Lumiere picture of people leaving a show of their pictures.

[877615] Panoramic View of Frere Camp Taken from the Front of an Armoured Train, November 29th, 1899
[DUPLICATE of the other 'Panoramic View of Frere Camp']



In [15]:
def strip_duplicate_flag(text):
    if pd.isna(text):
        return text
    # Remove "[DUPLICATE of '...']" or "[DUPLICATE of ...]" bracketed form
    text = re.sub(r"\[DUPLICATE\s+of\s+[^\]]*\]", '', text)
    # Remove "DUPLICATE ENTRY! See X" / "DUPLICATE RECORD: See X" trailing form
    text = re.sub(r"DUPLICATE\s*(?i:entry|record)[!:]?\s*See\s+.*", '', text)
    return text.strip()

cleaned_df.loc[broad_mask, 'overview'] = cleaned_df.loc[broad_mask, 'overview'].apply(strip_duplicate_flag)

# Check the result
for idx, row in cleaned_df[broad_mask][['title', 'overview']].iterrows():
    print(f"[{idx}] {row['title']}\n'{row['overview']}'\n")

[10430] A New Journey: Live at Slane Castle
''

[234643] Baijin long
'The first Cantonese talkie. It was produced in Shanghai (Tianyi Studio) and was based on a Cantonese opera play.'

[531552] Portsmouth
'Portsmouth training'

[606443] Audience Leaving Circus Carré After the Biograph Show in Amsterdam
'While he was trying, Dickson made the same sort of movie that others had before him. This is an imitation of a Lumiere picture of people leaving a show of their pictures.'

[877615] Panoramic View of Frere Camp Taken from the Front of an Armoured Train, November 29th, 1899
''



In [16]:
whitespace = cleaned_df.select_dtypes(include="object").apply(
    lambda col: col.str.strip().eq("").sum()
)

print(whitespace[whitespace > 0])

title         5
overview    187
dtype: int64


In [17]:
mask = cleaned_df["title"].str.strip().eq("")

cleaned_df[mask]

,title,original_title,release_date,original_language,genres,production_companies,production_countries,spoken_languages,cast,director,writers,producers,music_composer,keywords,overview,poster_path
155627,,Een Gulden eeuw-de kunst der Vlaamse primitieven,1954-01-01,nl,Documentary,Kunst en Kino,Belgium,Nederlands,"Jozef Kadijk, Johan Gemmeke",Paul Haesaerts,NaN,NaN,NaN,NaN,"Following the success of Rubens and his series of short art films, Paul Hasaerts embarked on a project exploring the Flemish Primitives. With this film, he set out to offer a sweeping overview of fifteenth-century Flemish painting, showcasing masterpieces by Van Eyck, Van der Weyden, Bouts, Van der Goes, Memling, Quinten Metsys, and Brueghel, capturing both their artistic brilliance and historical significance.",/xmB2nN4ZYMcS3LLWZgzFwGM7aLN.jpg
576938,,Своя чужая,NaT,ru,Drama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
985183,,Rekordzista,1962-03-06,pl,NaN,NaN,NaN,Polski,"Jan Bunda, Danuta Starczewska, Bogusz Bilewski",Stefan Matyjaszkiewicz,Jerzy Suszko,NaN,NaN,NaN,NaN,/Tr2v6KZHfOfg0EmqiPfdvVcSLI.jpg
1017833,,Дом,NaT,ru,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
1162324,,Homeless Tour Documentary,2026-05-09,en,"Documentary, Music",Anami Studios,Nigeria,NaN,"Lanre Masha, Sunday Matthew, Feyi, Jemadafe Caleb, Afolayan Oluwatomiwa Oluwanifemi, Bizzle Osikoya, Mrs Attah, Tolu Daniels, Oluwatosin Tiamiyu, Olabiyi Olamide, Ifeoluwa Jim Falola, Claire Otuya, ABDULAHI OLAWALE IBRAHIM",Ifeoluwa Jim Falola,NaN,"Olabiyi Olamide, Ifeoluwa Jim Falola",NaN,NaN,"Follows Afropop artist Llona through a first-of-its-kind 25-city tour across Nigeria with his deeply moving, storied debut album, Homeless. A feat that superstars such as Wizkid, Davido and Burna Boy are yet to achieve. This documentary answers how Llona pulled this off with his debut album.",/e0xvDXWkYlOAjuZyXe30VDp7Dgn.jpg


In [18]:
print(cleaned_df["title"].isna().sum())
print(cleaned_df["overview"].isna().sum())

0
163810


In [19]:
for col in ["title", "overview"]:
    mask = cleaned_df[col].str.strip().eq("")
    cleaned_df.loc[mask, col] = np.nan

In [20]:
whitespace = cleaned_df.select_dtypes(include="object").apply(
    lambda col: col.str.strip().eq("").sum()
)

print(whitespace[whitespace > 0])

Series([], dtype: int64)


In [21]:
print(cleaned_df["title"].isna().sum())
print(cleaned_df["overview"].isna().sum())

5
163997


# Check Duplicates

In [22]:
cleaned_df.shape


(1192422, 16)

In [23]:
cleaned_df.dtypes

title                           object
original_title                  object
release_date            datetime64[ns]
original_language               object
genres                          object
production_companies            object
production_countries            object
spoken_languages                object
cast                            object
director                        object
writers                         object
producers                       object
music_composer                  object
keywords                        object
overview                        object
poster_path                     object
dtype: object

In [24]:
print("release_date empty count:" , int(cleaned_df["release_date"].isna().sum()))
print("overview empty count:" , int(cleaned_df["overview"].isna().sum()))

release_date empty count: 100259
overview empty count: 163997


In [25]:
print(cleaned_df["original_title"].isna().sum())

0


## Remove Duplicates

In [26]:
# Build one key: title + date if available, else just title
cleaned_df["_dedup_key"] = (
    cleaned_df["original_title"].astype(str) + "||" +
    cleaned_df["release_date"].fillna("").astype(str)
)

# Detect
total_duplicates = cleaned_df.duplicated(subset=["_dedup_key"], keep="first").sum()
print("Total duplicate rows:", total_duplicates)

Total duplicate rows: 5601


In [27]:
# Remove
cleaned_df = cleaned_df.drop_duplicates(subset=["_dedup_key"], keep="first")
cleaned_df = cleaned_df.drop(columns=["_dedup_key"])

In [28]:
# Build one key: title + date if available, else just title
cleaned_df["_dedup_key"] = (
    cleaned_df["original_title"].astype(str) + "||" +
    cleaned_df["release_date"].fillna("").astype(str)
)

# Detect
total_duplicates = cleaned_df.duplicated(subset=["_dedup_key"], keep="first").sum()
print("Total duplicate rows:", total_duplicates)
cleaned_df = cleaned_df.drop(columns=["_dedup_key"])
cleaned_df.shape

Total duplicate rows: 0


(1186821, 16)

In [69]:
# cleaned_df.to_csv(mount_path + "/data/cleanedmovies.csv", index=False)

In [29]:
# First, look at raw values to see the actual delimiter format
for col in ["production_companies", "production_countries", "spoken_languages"]:
    print(col)
    print(cleaned_df[col].dropna().sample(5, random_state=42).tolist())
    print()

production_companies
['Greek Film Centre', 'Central Television USSR', 'Lebanese American University', 'Merchant Ivory Productions', 'HESTIA Media Prod, Synoptico']

production_countries
['Denmark', 'United States of America', 'Estonia', 'United Kingdom', 'South Africa']

spoken_languages
['English', 'English', 'తెలుగు', 'Français', 'English']



# DB Upload with `to_sql` (Run in Local)

## Part 1

In [ ]:
# cleaned_df = pd.read_csv("../data/cleanedmovies.csv")
# cleaned_df

In [30]:
def split_comma_list(s):
    parts = re.split(r',(?!\s*(?:Jr|jr|Sr|sr|II|III|IV|V|LTD|LLC|INC|Ltd|Inc|ltd)\.?\s*(?:,|$))\s*', s)
    return [p.strip() for p in parts if p.strip()]

In [31]:
cleaned_df['id'] = cleaned_df.index + 1
cleaned_df['genres'] = cleaned_df['genres'].fillna('').apply(
    lambda s: [g.strip() for g in s.split(',') if g.strip()]
)
cleaned_df['production_companies'] = cleaned_df['production_companies'].fillna('').apply(split_comma_list)
cleaned_df['production_countries'] = cleaned_df['production_countries'].fillna('').apply(
    lambda s: [g.strip() for g in s.split(',') if g.strip()]
)
cleaned_df['spoken_languages'] = cleaned_df['spoken_languages'].fillna('').apply(
    lambda s: [g.strip() for g in s.split(',') if g.strip()]
)
cleaned_df['cast_members'] = cleaned_df['cast'].fillna('').apply(split_comma_list)
cleaned_df['director'] = cleaned_df['director'].fillna('').apply(split_comma_list)
cleaned_df['writers'] = cleaned_df['writers'].fillna('').apply(split_comma_list)
cleaned_df['producers'] = cleaned_df['producers'].fillna('').apply(split_comma_list)
cleaned_df['music_composer'] = cleaned_df['music_composer'].fillna('').apply(split_comma_list)
cleaned_df['keywords'] = cleaned_df['keywords'].fillna('').apply(
    lambda s: [g.strip() for g in s.split('|') if g.strip()]
)
cleaned_df = cleaned_df.drop(columns=['cast'])

In [32]:
list_cols = [
    'genres', 'production_companies', 'production_countries',
    'spoken_languages', 'cast_members', 'director',
    'writers', 'producers', 'music_composer', 'keywords'
]

for col in list_cols:
    has_dupes = cleaned_df[col].apply(lambda lst: len(lst) != len(set(lst)))
    print(f"{col}: {has_dupes.sum()} rows with duplicate elements")

genres: 113 rows with duplicate elements
production_companies: 420 rows with duplicate elements
production_countries: 2 rows with duplicate elements
spoken_languages: 0 rows with duplicate elements
cast_members: 37 rows with duplicate elements
director: 2 rows with duplicate elements
writers: 1 rows with duplicate elements
producers: 3 rows with duplicate elements
music_composer: 1 rows with duplicate elements
keywords: 0 rows with duplicate elements


In [33]:
def dedupe_preserve_order(lst):
    seen = set()
    result = []
    for item in lst:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

for col in list_cols:
    cleaned_df[col] = cleaned_df[col].apply(dedupe_preserve_order)

In [34]:
list_cols = [
    'genres', 'production_companies', 'production_countries',
    'spoken_languages', 'cast_members', 'director',
    'writers', 'producers', 'music_composer', 'keywords'
]

for col in list_cols:
    has_dupes = cleaned_df[col].apply(lambda lst: len(lst) != len(set(lst)))
    print(f"{col}: {has_dupes.sum()} rows with duplicate elements")

genres: 0 rows with duplicate elements
production_companies: 0 rows with duplicate elements
production_countries: 0 rows with duplicate elements
spoken_languages: 0 rows with duplicate elements
cast_members: 0 rows with duplicate elements
director: 0 rows with duplicate elements
writers: 0 rows with duplicate elements
producers: 0 rows with duplicate elements
music_composer: 0 rows with duplicate elements
keywords: 0 rows with duplicate elements


In [35]:
for col in list_cols:
    exact = cleaned_df[col].apply(lambda lst: len(lst) != len(set(lst)))
    ci = cleaned_df[col].apply(lambda lst: len(lst) != len({str(x).lower() for x in lst}))
    print(f"{col}: exact={exact.sum()}, case-insensitive={ci.sum()}, delta={ci.sum() - exact.sum()}")

genres: exact=0, case-insensitive=0, delta=0
production_companies: exact=0, case-insensitive=15, delta=15
production_countries: exact=0, case-insensitive=0, delta=0
spoken_languages: exact=0, case-insensitive=0, delta=0
cast_members: exact=0, case-insensitive=19, delta=19
director: exact=0, case-insensitive=0, delta=0
writers: exact=0, case-insensitive=3, delta=3
producers: exact=0, case-insensitive=1, delta=1
music_composer: exact=0, case-insensitive=0, delta=0
keywords: exact=0, case-insensitive=0, delta=0


In [36]:
mask = cleaned_df['production_companies'].apply(
    lambda lst: len(lst) != len({str(x).lower() for x in lst})
)
cleaned_df.loc[mask, 'production_companies'].head(17).tolist()

[['Culver Entertainment',
  'Full On Service',
  'Full on Service',
  'Vivid White Productions',
  "Where's Lunch"],
 ['Tsk', 'tsk', 'tsk Productions'],
 ['The animation Workshop', 'The Animation Workshop'],
 ['The animation Workshop', 'The Animation Workshop'],
 ['The animation Workshop', 'The Animation Workshop'],
 ['And Film', 'Tsk', 'tsk', 'tsk Productions'],
 ['Blind Wink Productions',
  'VMI Worldwide',
  'VMI WorldWide',
  'VMI Productions',
  'Decal'],
 ['Isadice', 'ISADICE'],
 ['ID Films', 'id films', 'Investigation Discovery'],
 ['PASSWORD FILMS', 'password films'],
 ['filimo', 'Filimo'],
 ['Universidad del Cine', 'Amores Analógicos', 'Universidad Del Cine'],
 ['Drôle de Trame', 'Drôle de trame'],
 ['Roche Productions',
  'Fondation pour la Mémoire de la Shoah',
  'CNC',
  'Ministère des Armées',
  'Direction de la Mémoire',
  'de la Culture et des Archives',
  "Ministère des Armées - Secrétariat général pour l'administration",
  'Ministère des Armées - Direction de la mémoir

In [37]:
mask = cleaned_df['cast_members'].apply(
    lambda lst: len(lst) != len({str(x).lower() for x in lst})
)
cleaned_df.loc[mask, 'cast_members'].head(5).tolist()

[['Kim A-joong',
  'Kim Seung-o',
  'Jeong Soo-young',
  'Kim Joon-ho',
  'Shin So-yul',
  'Kwak Ji-min',
  'Kang Kyung-joon',
  'Jeong Hyeong-Ho',
  'Ji Sung',
  'Ok Ji-young',
  'Yoon Joo',
  'Kim Joo-ryoung',
  'Shin Hae-Chul',
  'Choi Eun-suk',
  'Kim Bo-mi',
  'Moon Ji-yoon',
  'Shin Hae-chul',
  'Kim Bo-yeon'],
 ['daniel', 'Daniel'],
 ['Shauna Macdonald',
  'Kim Roberts',
  'Christopher Russell',
  'Chris Gillett',
  'Danny Smith',
  'Helen Colliander',
  'Autumn Reeser',
  'Patrice Goodman',
  'Damon Runyan',
  'Shauna MacDonald',
  'Richard Burgi',
  'Neil Crone'],
 ['Ji An',
  'Shin Won-ho',
  'Song Jae-ho',
  'On Ju-wan',
  'Ji an',
  'Kwon Kyong-ha',
  'Jo Yoon-seo',
  'Kim Hye-ja',
  'Lee Sung-deuk',
  'Kim Seung-hyun',
  'Yun Jae-ho',
  'An Hye-gyeong',
  'Heo Jin',
  'Kim Ji-sung'],
 ['Yoshio Shin',
  'Yukina AOYAMA',
  'Nanami SHIRAKAWA',
  'Ryuichi Yoshida',
  'Aoi Ibuki',
  'aoi ibuki']]

In [38]:
def dedupe_case_insensitive_preserve_order(lst):
    seen = set()
    result = []
    for item in lst:
        key = str(item).lower()
        if key not in seen:
            seen.add(key)
            result.append(item)  # keep original casing of first occurrence
    return result

affected_cols = ['production_companies', 'cast_members', 'writers', 'producers']

for col in affected_cols:
    cleaned_df[col] = cleaned_df[col].apply(dedupe_case_insensitive_preserve_order)

In [39]:
for col in affected_cols:
    ci = cleaned_df[col].apply(lambda lst: len(lst) != len({str(x).lower() for x in lst}))
    print(f"{col}: {ci.sum()} remaining")

production_companies: 0 remaining
cast_members: 0 remaining
writers: 0 remaining
producers: 0 remaining


In [40]:
cleaned_df

,title,original_title,release_date,original_language,genres,production_companies,production_countries,spoken_languages,director,writers,producers,music_composer,keywords,overview,poster_path,id,cast_members
0,Ariel,Ariel,1988-10-21,fi,"[Drama, Romance, Crime]",[Villealfa Filmproductions],[Finland],[suomi],[Aki Kaurismäki],[Aki Kaurismäki],[Aki Kaurismäki],[],"[prison, underdog, helsinki, finland, factory worker, prisoner, falling in love]",A Finnish man goes to the city to find a job after the mine where he worked is closed and his father commits suicide.,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,1,"[Turo Pajala, Susanna Haavisto, Matti Pellonpää, Eetu Hilkamo, Erkki Pajala, Matti Jaaranen, Hannu Viholainen, Jorma Markkula, Tarja Keinänen, Eino Kuusela, Kauko Laalo, Jyrki Olsonen, Esko Nikkari, Marja Packalén, Mikko Remes, Tomi Salmela, Reijo Marin, Heikki Salomaa, Veikko Uusimäki, Esko Salminen, Hannu Kivisalo, Pekka Wilen, Heikki Anttila, Pentti Auer, Timo Harakka, Kari Helaseppä, Juuso Hirvikangas, Hanna Jokinen, Sakari Kuosmanen, Sami Lanki, Jouko Lumme, Mikko Lyytikäinen, Timo Markko, Merja Pulkkinen, Markku Rantala, Sirkka Rautiainen, Jaakko Talaskivi, Timo Toikka, Olli Varja]"
1,Shadows in Paradise,Varjoja paratiisissa,1986-10-17,fi,"[Drama, Romance]",[Villealfa Filmproductions],[Finland],"[svenska, suomi, English]",[Aki Kaurismäki],[Aki Kaurismäki],[Mika Kaurismäki],[],"[helsinki, finland, salesclerk, garbage]","Nikander, a rubbish collector and would-be entrepreneur, finds his plans for success dashed when his business associate dies. One evening, he meets Ilona, a down-on-her-luck cashier, in a local supermarket. Falteringly, a bond begins to develop between them.",/nj01hspawPof0mJmlgfjuLyJuRN.jpg,2,"[Matti Pellonpää, Kati Outinen, Sakari Kuosmanen, Esko Nikkari, Kylli Köngäs, Pekka Laiho, Jukka-Pekka Palo, Svante Korkiakoski, Mari Rantasila, Safka Pekkonen, Antti Ortamo, Mato Valtonen, Sakke Järvenpää, Ulla Kuosmanen, Neka Haapanen, Pentti Koski, Ari Korhonen, Teuvo Rissanen, Erkki Rissanen, Sirkka Silin, Marina Martinoff, Jussi Tiitinen, Riikka Kuosmanen, Eskil Mansikka, Jaakko Talaskivi, Haije Alanoja, Bertta Pellonpää, Helmeri Pellonpää, Malla Hukkanen, Aki Kaurismäki, Olli Varja, Tanja Talaskivi, Jukka Mäkinen]"
2,Four Rooms,Four Rooms,1995-12-09,en,[Comedy],"[Miramax, A Band Apart]",[United States of America],[English],"[Quentin Tarantino, Robert Rodriguez, Alexandre Rockwell, Allison Anders]","[Quentin Tarantino, Robert Rodriguez, Alexandre Rockwell, Allison Anders]","[Quentin Tarantino, Lawrence Bender, Alexandre Rockwell]",[Combustible Edison],"[hotel, new year's eve, witch, bet, sperm, hotel room, anthology, los angeles, california, hoodlum, multiple storylines, woman director]",It's Ted the Bellhop's first night on the job...and the hotel's very unusual guests are about to place him in some outrageous predicaments. It seems that this evening's room service is serving up one unbelievable happening after another.,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg,3,"[Sammi Davis, Salma Hayek Pinault, Lawrence Bender, Madonna, Jennifer Beals, Alicia Witt, Lili Taylor, Patricia Vonne, Laura Rush, Unruly Julie McClean, Quentin Tarantino, Danny Verduzco, Tamlyn Tomita, Ione Skye, Marisa Tomei, Amanda de Cadenet, Lana McKissack, Tim Roth, Kimberly Blair, Marc Lawrence, Paul Skemp, Antonio Banderas, Kathy Griffin, Bruce Willis, David Proval, Quinn Hellerman, Valeria Golino, Paul Calderon]"
3,Judgment Night,Judgment Night,1993-10-15,en,"[Action, Crime, Drama, Thriller]","[Largo Entertainment, JVC, Universal Pictures]",[United States of America],[English],[Stephen Hopkins],"[Lewis Colick, Jere Cunningham]","[Gene Levy, Marilyn Vance, Lloyd Segan]",[Alan Silvestri],"[drug dealer, chicago, illinois, escape, one night, boxing]","Four young friends, while taking a shortcut en route to a local boxing match, witness a brutal murder which leaves them running for their lives.",/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg,4,"[Emilio Estevez, Cuba Gooding Jr., Den

In [ ]:
# cleaned_df.to_csv(mount_path + "/data/dbsavemovies.csv", index=False)

In [41]:
both_missing = cleaned_df['title'].isna() & cleaned_df['original_title'].isna()
print(both_missing.sum())

0


In [42]:
import re

# Assumes cleaned_df with an 'overview' column (adjust name if different)
sample = cleaned_df['overview'].dropna().astype(str)

# Check 1: HTML entities present?
html_entity_count = sample.str.contains(r'&[a-zA-Z#0-9]+;', regex=True).sum()

# Check 2: HTML tags present?
html_tag_count = sample.str.contains(r'<[^>]+>', regex=True).sum()

# Check 3: Non-breaking spaces / zero-width chars?
nbsp_count = sample.str.contains('\xa0').sum()
zwsp_count = sample.str.contains('\u200b').sum()

# Check 4: Excess whitespace / newlines?
whitespace_count = sample.str.contains(r'\s{2,}|\n|\t', regex=True).sum()

# Check 5: Empty or near-empty overviews?
empty_count = (sample.str.strip().str.len() == 0).sum()
very_short_count = (sample.str.strip().str.len() < 20).sum()

# Check 6: Length distribution (word count, for the MiniLM 256-token concern)
word_counts = sample.str.split().str.len()

print(f"Total overviews checked: {len(sample)}")
print(f"HTML entities found in: {html_entity_count} rows")
print(f"HTML tags found in: {html_tag_count} rows")
print(f"Non-breaking spaces in: {nbsp_count} rows")
print(f"Zero-width spaces in: {zwsp_count} rows")
print(f"Excess whitespace/newlines in: {whitespace_count} rows")
print(f"Empty overviews: {empty_count}")
print(f"Very short (<20 chars) overviews: {very_short_count}")
print(f"\nWord count distribution:\n{word_counts.describe()}")
print(f"Overviews >256 words (likely truncated by MiniLM): {(word_counts > 256).sum()}")

Total overviews checked: 1023485
HTML entities found in: 2173 rows
HTML tags found in: 17 rows
Non-breaking spaces in: 957 rows
Zero-width spaces in: 1240 rows
Excess whitespace/newlines in: 63006 rows
Empty overviews: 0
Very short (<20 chars) overviews: 13295

Word count distribution:
count    1.023485e+06
mean     4.511022e+01
std      3.384341e+01
min      1.000000e+00
25%      2.000000e+01
50%      3.600000e+01
75%      6.200000e+01
max      2.310000e+02
Name: overview, dtype: float64
Overviews >256 words (likely truncated by MiniLM): 0


In [43]:
import re
import html

def clean_overview(text: str) -> str:
    """
    Clean plot/overview text for sentence-transformer embedding.
    Scoped to what's ACTUALLY present in this dataset (per diagnostic):
    - 6.2% rows: excess whitespace/newlines -> collapse
    - ~0.2% rows: HTML entities/tags -> strip
    - ~0.2% rows: NBSP/zero-width chars -> strip
    Preserves sentence structure, casing, punctuation.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    text = html.unescape(text)                    # &amp; -> &
    text = re.sub(r"<[^>]+>", " ", text)           # strip stray HTML tags
    text = text.replace("\xa0", " ").replace("\u200b", "")  # NBSP / zero-width
    text = re.sub(r"\s+", " ", text)               # collapse whitespace/newlines
    text = text.strip()

    return text


# Apply and flag thin overviews in the same pass
cleaned_df['overview_clean'] = cleaned_df['overview'].apply(clean_overview)
cleaned_df['has_thin_overview'] = cleaned_df['overview_clean'].str.len() < 20

# Verify: before/after on rows that actually had junk
dirty_mask = (
    cleaned_df['overview'].str.contains(r'&[a-zA-Z#0-9]+;', regex=True, na=False) |
    cleaned_df['overview'].str.contains(r'\s{2,}|\n|\t', regex=True, na=False) |
    cleaned_df['overview'].str.contains('\xa0', na=False) |
    cleaned_df['overview'].str.contains('\u200b', na=False)
)

sample = cleaned_df[dirty_mask].head(5)
for idx, row in sample.iterrows():
    print("BEFORE:", repr(row['overview'][:150]))
    print("AFTER: ", repr(row['overview_clean'][:150]))
    print("-" * 80)

print(f"\nTotal thin overviews flagged: {cleaned_df['has_thin_overview'].sum()}")

BEFORE: 'In an attempt to pull her family together, Adèlle travels with her young daughter Sarah to Wales to visit her father. The morning after they arrive, S'
AFTER:  'In an attempt to pull her family together, Adèlle travels with her young daughter Sarah to Wales to visit her father. The morning after they arrive, S'
--------------------------------------------------------------------------------
BEFORE: "Based on Michel Houellebecq's controversial novel, Atomised (aka The Elementary Particles) focuses on Michael and Bruno, two very different half-broth"
AFTER:  "Based on Michel Houellebecq's controversial novel, Atomised (aka The Elementary Particles) focuses on Michael and Bruno, two very different half-broth"
--------------------------------------------------------------------------------
BEFORE: 'As armies mass for a final battle that will decide the fate of the world--and powerful, ancient forces of Light and Dark compete to determine the outc'
AFTER:  'As armies mass for a fin

In [44]:
sample2 = cleaned_df[dirty_mask].sample(2)
for idx, row in sample2.iterrows():
    if row['overview'] != row['overview_clean']:
        print("DIFFERS")
        print("BEFORE:", repr(row['overview']))
        print("AFTER: ", repr(row['overview_clean']))
    else:
        print("IDENTICAL (mask false positive or fix outside preview window)")
    print("-" * 80)

DIFFERS
BEFORE: 'TIG welding has been long recognized as the premiere welding process, which can make beautiful, strong welds on nearly any metal. In this easy-to-follow program, Ron Covell covers the basic processes used for TIG welding, covering proper machine setup, joint preparation, electrode selection and preparation, torch technique, and perhaps most importantly, what to check for when things go wrong!\r Ron’s thoughtful description of the welding process, coupled with the close-up photography that brings the action into clear view makes the seemingly difficult process of TIG welding easy to understand.'
AFTER:  'TIG welding has been long recognized as the premiere welding process, which can make beautiful, strong welds on nearly any metal. In this easy-to-follow program, Ron Covell covers the basic processes used for TIG welding, covering proper machine setup, joint preparation, electrode selection and preparation, torch technique, and perhaps most importantly, what to check fo

In [45]:
# Drop the raw column, promote clean to be the only "overview"
cleaned_df['overview'] = cleaned_df['overview_clean']
cleaned_df = cleaned_df.drop(columns=['overview_clean'])

In [46]:
print(f"Total rows in cleaned_df: {len(cleaned_df)}")
print(f"Null overviews: {cleaned_df['overview'].isna().sum()}")
print(f"Empty string overviews: {(cleaned_df['overview'] == '').sum()}")
print(f"Whitespace-only overviews: {cleaned_df['overview'].str.strip().eq('').sum()}")

Total rows in cleaned_df: 1186821
Null overviews: 0
Empty string overviews: 163337
Whitespace-only overviews: 163337


## Part 2

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username="nd2002",
    password="As123@we",
    host="localhost",
    port=15432,
    database="moviedb"
)

engine = create_engine(url)

DDL = """
CREATE TABLE IF NOT EXISTS movies (
    id INTEGER PRIMARY KEY,
    title TEXT,
    original_title TEXT,
    release_date DATE,
    original_language TEXT,
    genres TEXT[],
    production_companies TEXT[],
    production_countries TEXT[],
    spoken_languages TEXT[],
    cast_members TEXT[],
    director TEXT[],
    writers TEXT[],
    producers TEXT[],
    music_composer TEXT[],
    keywords TEXT[],
    poster_path TEXT,
    overview TEXT,
    has_thin_overview BOOLEAN NOT NULL DEFAULT TRUE,
    CONSTRAINT chk_title_or_original_title CHECK (
        title IS NOT NULL OR original_title IS NOT NULL
    )
);

CREATE INDEX IF NOT EXISTS idx_movies_genres ON movies USING GIN (genres);
CREATE INDEX IF NOT EXISTS idx_movies_cast ON movies USING GIN (cast_members);
CREATE INDEX IF NOT EXISTS idx_movies_keywords ON movies USING GIN (keywords);
CREATE INDEX IF NOT EXISTS idx_movies_director ON movies USING GIN (director);
CREATE INDEX IF NOT EXISTS idx_movies_title ON movies (title);
CREATE INDEX IF NOT EXISTS idx_movies_lang ON movies (original_language);

CREATE TABLE IF NOT EXISTS tag_lookup (
    tag_type TEXT NOT NULL,
    name TEXT NOT NULL,
    movie_count INTEGER NOT NULL,
    PRIMARY KEY (tag_type, name)
);

TRUNCATE TABLE tag_lookup;

INSERT INTO tag_lookup (tag_type, name, movie_count)
SELECT t, x, COUNT(DISTINCT id)
FROM (
  SELECT id, 'genre'             AS t, unnest(genres)               AS x FROM movies
  UNION ALL SELECT id, 'original_language',  original_language           FROM movies
  UNION ALL SELECT id, 'cast',               unnest(cast_members)        FROM movies
  UNION ALL SELECT id, 'director',           unnest(director)            FROM movies
  UNION ALL SELECT id, 'writer',             unnest(writers)             FROM movies
  UNION ALL SELECT id, 'producer',           unnest(producers)           FROM movies
  UNION ALL SELECT id, 'composer',           unnest(music_composer)      FROM movies
  UNION ALL SELECT id, 'keyword',            unnest(keywords)            FROM movies
  UNION ALL SELECT id, 'company',            unnest(production_companies) FROM movies
  UNION ALL SELECT id, 'country',            unnest(production_countries) FROM movies
  UNION ALL SELECT id, 'spoken_language',    unnest(spoken_languages)    FROM movies
) s
WHERE x IS NOT NULL AND x <> ''
GROUP BY t, x;

"""

with engine.begin() as conn:
    conn.execute(text(DDL))

In [ ]:
cleaned_df.to_sql(
    'movies',
    engine,
    if_exists='append',
    index=False,
    method='multi',
    chunksize=5000
)

1186821

# Build tag soup per movie

In [54]:
# ARRAY_TAG_FIELDS = [
#     'genres', 'production_companies', 'production_countries',
#     'spoken_languages', 'cast_members', 'director',
#     'writers', 'producers', 'music_composer', 'keywords',
# ]

def normalize_token(item) -> str:
    if item is None:
        return ''
    if isinstance(item, (list, tuple)):
        return ''.join(normalize_token(x) for x in item)
    text = str(item).lower().strip()
    return re.sub(r'[^a-z0-9]', '', text)

# def build_tag_soup(row) -> str:
#     tokens = []
#     for field in ARRAY_TAG_FIELDS:
#         value = row.get(field)
#         if value is None:
#             continue
#         if isinstance(value, (list, tuple)):
#             tokens.extend(normalize_token(v) for v in value)
#         else:
#             tokens.append(normalize_token(value))
#     return ' '.join(t for t in tokens if t)

# cleaned_df['tag_soup'] = cleaned_df.apply(build_tag_soup, axis=1)

In [55]:
import re

def normalize_token(item) -> str:
    if item is None:
        return ''
    if isinstance(item, (list, tuple)):
        return ''.join(normalize_token(x) for x in item)
    text = str(item).lower().strip()
    return re.sub(r'[^a-z0-9]', '', text)

def field_token_stats(df, field):
    """Count total tokens and unique tokens contributed by one field."""
    all_tokens = []
    for value in df[field]:
        if value is None:
            continue
        if isinstance(value, (list, tuple)):
            all_tokens.extend(normalize_token(v) for v in value if v)
        else:
            tok = normalize_token(value)
            if tok:
                all_tokens.append(tok)
    return {
        'field': field,
        'total_tokens': len(all_tokens),
        'unique_tokens': len(set(all_tokens)),
    }

fields_to_check = [
    'genres', 'production_companies', 'production_countries',
    'spoken_languages', 'cast_members', 'director',
    'writers', 'producers', 'music_composer', 'keywords'
]

results = [field_token_stats(cleaned_df, f) for f in fields_to_check]

import pandas as pd
stats_df = pd.DataFrame(results).sort_values('unique_tokens', ascending=False)
print(stats_df.to_string(index=False))

               field  total_tokens  unique_tokens
        cast_members       7634174        2160434
            director       1148159         429223
             writers        989703         405550
           producers        874993         349330
production_companies        900861         206940
            keywords       1239155          73792
      music_composer        164091          47060
production_countries        839040            250
    spoken_languages        854097             63
              genres       1356290             19


In [56]:
import re
import numpy as np
import joblib
from scipy.sparse import hstack, save_npz
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD

def normalize_token(item) -> str:
    if item is None:
        return ''
    if isinstance(item, (list, tuple)):
        return ''.join(normalize_token(x) for x in item)
    text = str(item).lower().strip()
    return re.sub(r'[^a-z0-9]', '', text)

def build_tag_soup(row, fields) -> str:
    tokens = []
    for field in fields:
        value = row.get(field)
        if value is None:
            continue
        if isinstance(value, (list, tuple)):
            tokens.extend(normalize_token(v) for v in value)
        else:
            tokens.append(normalize_token(value))
    return ' '.join(t for t in tokens if t)


# Regrouped by actual unique-token count, not field-name intuition
HIGH_FIELDS = ['cast_members','genres','spoken_languages','production_countries']
MID_FIELDS  = ['producers','writers','music_composer']
LOW_FIELDS  = ['director','keywords']

# Build soups
high_soup = cleaned_df.apply(lambda row: build_tag_soup(row, HIGH_FIELDS), axis=1)
mid_soup  = cleaned_df.apply(lambda row: build_tag_soup(row, MID_FIELDS), axis=1)
low_soup  = cleaned_df.apply(lambda row: build_tag_soup(row, LOW_FIELDS), axis=1)

# Vectorize each tier separately with its own min_df
high_vectorizer = CountVectorizer(min_df=25, tokenizer=str.split, lowercase=False, token_pattern=None)
X_high = high_vectorizer.fit_transform(high_soup)

mid_vectorizer = CountVectorizer(min_df=15, tokenizer=str.split, lowercase=False, token_pattern=None)
X_mid = mid_vectorizer.fit_transform(mid_soup)

low_vectorizer = CountVectorizer(min_df=10, tokenizer=str.split, lowercase=False, token_pattern=None)
X_low = low_vectorizer.fit_transform(low_soup)

X = hstack([X_high, X_mid, X_low]).tocsr()
print(f"High vocab: {X_high.shape[1]}, Mid vocab: {X_mid.shape[1]}, Low vocab: {X_low.shape[1]}")
print(f"Combined shape: {X.shape}")

# Empty-row check — carried over from earlier, still worth confirming on this matrix
row_sums = np.asarray(X.sum(axis=1)).flatten()
empty_tag_rows = (row_sums == 0).sum()
print(f"Empty tag rows: {empty_tag_rows} out of {X.shape[0]}")

# Fit SVD once, on the correct combined matrix
svd = TruncatedSVD(n_components=300, random_state=42)
X_reduced = svd.fit_transform(X)
print(f"Explained variance ratio (cumulative): {svd.explained_variance_ratio_.sum():.4f}")

High vocab: 49707, Mid vocab: 17107, Low vocab: 32302
Combined shape: (1186821, 99116)
Empty tag rows: 122901 out of 1186821
Explained variance ratio (cumulative): 0.4027


In [57]:
import os

os.makedirs('artifacts', exist_ok=True)

# 1. Vectorizers — three, not one. Rename to reflect the split architecture.
joblib.dump(high_vectorizer, 'tag_vectorizer_high.joblib')
joblib.dump(mid_vectorizer,  'tag_vectorizer_mid.joblib')
joblib.dump(low_vectorizer,  'tag_vectorizer_low.joblib')

# 2. Movie IDs — MUST be captured from the same cleaned_df used above, same order
movie_ids = cleaned_df['id'].values  # adjust column name if not 'id'
np.save('tag_matrix_movie_ids.npy', movie_ids)

# 3. Sparse tag matrix (exact scoring, Stage 2)
save_npz('tag_matrix.npz', X)

# 4. Fitted SVD model (needed to reduce query vectors at runtime)
joblib.dump(svd, 'tag_svd_model.joblib')

# 5. Dense SVD-reduced matrix (feeds FAISS HNSW build, next step)
np.save('tag_matrix_svd_reduced.npy', X_reduced)

# Sanity check before you trust any of it
assert X.shape[0] == len(movie_ids) == X_reduced.shape[0], "Row count mismatch — do not proceed to FAISS"

In [62]:
# 6. has_tags mask — computed here while X and movie_ids are guaranteed aligned
has_tags_mask = np.asarray(X.sum(axis=1)).flatten() > 0  # True where row has ≥1 nonzero tag entry
np.save('has_tags_mask.npy', has_tags_mask)

assert has_tags_mask.shape[0] == len(movie_ids), "Mask/id length mismatch — do not proceed"
print(f"{has_tags_mask.sum()} of {len(movie_ids)} movies have tags "
      f"({(~has_tags_mask).sum()} empty)")

1063920 of 1186821 movies have tags (122901 empty)


In [63]:
!dagshub upload --bucket dilminidasunika/movie_recommendation_system tag_vectorizer_high.joblib data/artifacts/tag_vectorizer_high.joblib
!dagshub upload --bucket dilminidasunika/movie_recommendation_system tag_vectorizer_mid.joblib data/artifacts/tag_vectorizer_mid.joblib
!dagshub upload --bucket dilminidasunika/movie_recommendation_system tag_vectorizer_low.joblib data/artifacts/tag_vectorizer_low.joblib

!dagshub upload --bucket dilminidasunika/movie_recommendation_system tag_matrix_movie_ids.npy data/artifacts/tag_matrix_movie_ids.npy
!dagshub upload --bucket dilminidasunika/movie_recommendation_system tag_matrix.npz data/artifacts/tag_matrix.npz

!dagshub upload --bucket dilminidasunika/movie_recommendation_system tag_svd_model.joblib data/artifacts/tag_svd_model.joblib
!dagshub upload --bucket dilminidasunika/movie_recommendation_system tag_matrix_svd_reduced.npy data/artifacts/tag_matrix_svd_reduced.npy
!dagshub upload --bucket dilminidasunika/movie_recommendation_system has_tags_mask.npy data/artifacts/has_tags_mask.npy

Accessing as Nuwantha97
Client created. Use the name of the repo (movie_recommendation_system) as the 
name of the bucket
Uploaded /kaggle/working/tag_vectorizer_high.joblib to 
s3://movie_recommendation_system/data/artifacts/tag_vectorizer_high.joblib
Accessing as Nuwantha97
Client created. Use the name of the repo (movie_recommendation_system) as the 
name of the bucket
Uploaded /kaggle/working/tag_vectorizer_mid.joblib to 
s3://movie_recommendation_system/data/artifacts/tag_vectorizer_mid.joblib
Accessing as Nuwantha97
Client created. Use the name of the repo (movie_recommendation_system) as the 
name of the bucket
Uploaded /kaggle/working/tag_vectorizer_low.joblib to 
s3://movie_recommendation_system/data/artifacts/tag_vectorizer_low.joblib
Accessing as Nuwantha97
Client created. Use the name of the repo (movie_recommendation_system) as the 
name of the bucket
Uploaded /kaggle/working/tag_matrix_movie_ids.npy to 
s3://movie_recommendation_system/data/artifacts/tag_matrix_movie_ids.

In [64]:
row_nnz = X.getnnz(axis=1)  # number of non-zero entries per row
empty_rows = np.where(row_nnz == 0)[0]

print(f"Total movies: {X.shape[0]}")
print(f"Empty tag vectors: {len(empty_rows)}")
print(f"Percentage: {len(empty_rows) / X.shape[0] * 100:.3f}%")


Total movies: 1186821
Empty tag vectors: 122901
Percentage: 10.355%


In [ ]:
from dagshub import get_repo_bucket_client
import scipy.sparse

client = get_repo_bucket_client("dilminidasunika/movie_recommendation_system", flavor="boto")

def load_npy(key):
    obj = client.get_object(
        Bucket="movie_recommendation_system",
        Key=key,
    )
    return np.load(io.BytesIO(obj["Body"].read()), allow_pickle=True)

In [ ]:
# from dagshub import get_repo_bucket_client
# import scipy.sparse

# client = get_repo_bucket_client("dilminidasunika/movie_recommendation_system", flavor="boto")

# obj = client.get_object(
#     Bucket="movie_recommendation_system",
#     Key="data/artifacts/tag_matrix.npz",
# )
# X = scipy.sparse.load_npz(io.BytesIO(obj["Body"].read()))
# print(X.shape)

In [ ]:
import chromadb
import numpy as np

# Load your existing artifacts
X_reduced = load_npy("artifacts/tag_matrix_svd_reduced.npy").astype('float32')
movie_ids = load_npy("artifacts/tag_matrix_movie_ids.npy")
has_tags_mask = load_npy("artifacts/has_tags_mask.npy")

print(X_reduced.shape)
print(movie_ids.shape)
print(has_tags_mask.shape)

# Exclude zero-tag rows BEFORE indexing — same reasoning as FAISS:
# a zero vector has no cosine direction and shouldn't be searchable
X_indexed = X_reduced[has_tags_mask]
ids_indexed = movie_ids[has_tags_mask]

assert X_indexed.shape[0] == len(ids_indexed), "Filtered row/id mismatch — do not proceed"
print(f"Indexing {len(ids_indexed)} of {len(movie_ids)} movies "
      f"({(~has_tags_mask).sum()} excluded, no tags)")

client = chromadb.PersistentClient(path="./chroma_tag_store")

tag_collection = client.create_collection(
    name="tag_vectors",
    metadata={"hnsw:space": "cosine"}
)

batch_size = 5000
for i in range(0, len(X_indexed), batch_size):
    batch_vecs = X_indexed[i:i+batch_size].tolist()
    batch_ids = [str(mid) for mid in ids_indexed[i:i+batch_size]]
    tag_collection.add(
        embeddings=batch_vecs,
        ids=batch_ids
    )

print(f"Indexed {tag_collection.count()} tag vectors")

In [59]:
import faiss
import numpy as np

X_reduced = X_reduced.astype('float32')  # FAISS requires float32, sklearn outputs float64

dimension = X_reduced.shape[1]  # 300
index = faiss.IndexHNSWFlat(dimension, 32)  # 32 = M, number of connections per node — a standard default
index.hnsw.efConstruction = 200  # build-time search depth — higher = better index quality, slower build

index.add(X_reduced)

faiss.write_index(index, 'tag_faiss_index.bin')
print(f"Indexed {index.ntotal} vectors")

Indexed 1186821 vectors


In [64]:
!dagshub upload --bucket dilminidasunika/movie_recommendation_system tag_faiss_index.bin data/artifacts/tag_faiss_index.bin

Accessing as Nuwantha97
Client created. Use the name of the repo (movie_recommendation_system) as the 
name of the bucket
Uploaded /kaggle/working/tag_faiss_index.bin to 
s3://movie_recommendation_system/data/artifacts/tag_faiss_index.bin


# Plot embeddings + Chroma setup

In [48]:
!pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 25.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 28.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 33.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38

In [49]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
assert device == 'cuda', "GPU not detected — check Kaggle accelerator setting before running full encode"

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [50]:
from sentence_transformers import SentenceTransformer
import numpy as np

has_overview = cleaned_df['overview'].str.strip().str.len() > 0

embed_df = cleaned_df[has_overview]
embed_ids = embed_df['id'].values
embed_texts = embed_df['overview']

embeddings = model.encode(embed_texts.tolist(), batch_size=256,
                           show_progress_bar=True, convert_to_numpy=True,
                           normalize_embeddings=True)

np.save("plot_embeddings.npy", embeddings.astype(np.float32))
np.save("plot_embeddings_movie_ids.npy", embed_ids)

Batches:   0%|          | 0/3998 [00:00<?, ?it/s]

In [51]:
!dagshub upload --bucket dilminidasunika/movie_recommendation_system plot_embeddings.npy data/artifacts/plot_embeddings.npy
!dagshub upload --bucket dilminidasunika/movie_recommendation_system plot_embeddings_movie_ids.npy data/artifacts/plot_embeddings_movie_ids.npy

Accessing as Nuwantha97
Client created. Use the name of the repo (movie_recommendation_system) as the 
name of the bucket
Uploaded /kaggle/working/plot_embeddings.npy to 
s3://movie_recommendation_system/data/artifacts/plot_embeddings.npy
Accessing as Nuwantha97
Client created. Use the name of the repo (movie_recommendation_system) as the 
name of the bucket
Uploaded /kaggle/working/plot_embeddings_movie_ids.npy to 
s3://movie_recommendation_system/data/artifacts/plot_embeddings_movie_ids.npy


In [ ]:
import chromadb

client = chromadb.PersistentClient(path="artifacts/chroma_db")
collection = client.create_collection(
    name="plot_embeddings",
    metadata={"hnsw:space": "cosine"}  # explicit, don't rely on Chroma's default
)

batch_size = 5000
for i in range(0, len(movie_ids), batch_size):
    batch_ids = [str(mid) for mid in movie_ids[i:i+batch_size]]  # Chroma ids must be strings
    batch_embeddings = embeddings[i:i+batch_size].tolist()
    collection.add(
        ids=batch_ids,
        embeddings=batch_embeddings
    )